# 06 - Sentiment Analysis delle recensioni Amazon con TF-IDF

In questo notebook costruiamo un classificatore binario per recensioni Amazon usando tecniche NLP classiche:

- parsing del formato fastText `__label__1 / __label__2`
- preprocessing testuale con pulizia del testo, stopword italiane e stemming opzionale
- vettorizzazione `TF-IDF`
- confronto fra `LogisticRegression`, `MultinomialNB`, `LinearSVC` e `SGDClassifier`
- tuning con `GridSearchCV`
- valutazione finale, error analysis e deployment con `joblib`

Obiettivi target:

- Accuracy > `85%`
- F1-score > `0.83`
- AUC-ROC > `0.90`

Nota metodologica importante:

- la pagina Kaggle del dataset `bittlingmayer/amazonreviews` indica che il corpus e' composto **soprattutto da recensioni in inglese**, con poche recensioni in altre lingue
- se il tuo obiettivo finale e' un modello strettamente **italiano**, piu' avanti trovi un filtro lingua opzionale oppure puoi sostituire il dataset con un corpus italiano dedicato

## Ambiente consigliato

Il notebook assume di trovare nella cartella corrente questi file:

- `train.ft.txt.bz2`
- `test.ft.txt.bz2`

Pacchetti consigliati:

```powershell
python -m pip install pandas numpy matplotlib seaborn scikit-learn wordcloud joblib nltk langdetect
```

Note:

- `langdetect` e' opzionale: serve solo per il filtro lingua
- `nltk` e' opzionale: serve solo se vuoi attivare lo stemming italiano
- il notebook e' stato preparato per essere eseguito **step by step**; non contiene output pre-calcolati

In [ ]:
import bz2
import random
import re
import warnings
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
except ImportError:
    WordCloud = None
    WORDCLOUD_AVAILABLE = False

try:
    from langdetect import DetectorFactory, LangDetectException, detect
    DetectorFactory.seed = 42
    LANGDETECT_AVAILABLE = True
except ImportError:
    DetectorFactory = None
    LangDetectException = Exception
    detect = None
    LANGDETECT_AVAILABLE = False

try:
    from nltk.stem.snowball import SnowballStemmer
    STEMMER_AVAILABLE = True
except ImportError:
    SnowballStemmer = None
    STEMMER_AVAILABLE = False

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

ROOT = Path.cwd()
TRAIN_PATH = ROOT / "train.ft.txt.bz2"
TEST_PATH = ROOT / "test.ft.txt.bz2"
ARTIFACTS_DIR = ROOT / "artifacts" / "06_sentiment_recensioni"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_SAMPLE_SIZE = 100_000
TEST_SAMPLE_SIZE = 40_000
VALID_SIZE = 0.20

TARGET_METRICS = {
    "accuracy": 0.85,
    "f1": 0.83,
    "roc_auc": 0.90,
}

ENABLE_LANGUAGE_FILTER = False
TARGET_LANGUAGE = "it"

for path in [TRAIN_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"File non trovato: {path}")

print("Cartella di lavoro:", ROOT.resolve())
print("Train dataset:", TRAIN_PATH.resolve())
print("Test dataset:", TEST_PATH.resolve())
print("WordCloud disponibile:", WORDCLOUD_AVAILABLE)
print("Language detection disponibile:", LANGDETECT_AVAILABLE)
print("Stemmer disponibile:", STEMMER_AVAILABLE)

## 1. Dataset preparation e contesto

Il dataset e' nel formato `fastText`:

```text
__label__1 testo recensione...
__label__2 testo recensione...
```

Mapping del target binario:

- `__label__1 -> 0 -> negativo` (recensioni da 1-2 stelle)
- `__label__2 -> 1 -> positivo` (recensioni da 4-5 stelle)

Per motivi di sviluppo rapido usiamo inizialmente:

- `100_000` campioni casuali dal train set
- un sottoinsieme del test set per evaluation piu' veloce

Il caricamento usa **reservoir sampling** per evitare di decomprimere l'intero file in memoria.

In [ ]:
FASTTEXT_PATTERN = re.compile(r"^(?P<label>__label__[12])\s+(?P<text>.+)$")

def parse_fasttext_line(line):
    line = line.strip()
    if not line:
        return None

    match = FASTTEXT_PATTERN.match(line)
    if match is None:
        return None

    raw_label = match.group("label")
    text = match.group("text").strip()
    target = 1 if raw_label == "__label__2" else 0

    return {
        "raw_label": raw_label,
        "target": target,
        "sentiment": "positivo" if target == 1 else "negativo",
        "text": text,
    }

def reservoir_sample_lines(file_path, sample_size=None, seed=42, max_lines=None):
    if sample_size is not None and sample_size <= 0:
        raise ValueError("sample_size deve essere positivo oppure None.")

    rng = random.Random(seed)
    reservoir = []
    seen = 0

    with bz2.open(file_path, mode="rt", encoding="utf-8", errors="ignore") as file:
        for raw_line in file:
            line = raw_line.strip()
            if not line:
                continue

            seen += 1
            if max_lines is not None and seen > max_lines:
                break

            if sample_size is None:
                reservoir.append(line)
                continue

            if len(reservoir) < sample_size:
                reservoir.append(line)
            else:
                replace_index = rng.randint(0, seen - 1)
                if replace_index < sample_size:
                    reservoir[replace_index] = line

    return reservoir

def load_fasttext_dataframe(file_path, sample_size=None, seed=42, max_lines=None):
    sampled_lines = reservoir_sample_lines(
        file_path=file_path,
        sample_size=sample_size,
        seed=seed,
        max_lines=max_lines,
    )

    rows = []
    for line in sampled_lines:
        parsed = parse_fasttext_line(line)
        if parsed is not None:
            rows.append(parsed)

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f"Nessuna riga valida trovata in {file_path}")

    df["source_file"] = file_path.name
    return df.reset_index(drop=True)

def summarize_dataset(df, name):
    summary = pd.DataFrame(
        {
            "count": df["target"].value_counts().sort_index(),
            "percentage": (df["target"].value_counts(normalize=True).sort_index() * 100).round(2),
        }
    )
    summary.index = summary.index.map({0: "negativo", 1: "positivo"})
    print(f"\n{name}")
    display(summary)
    print("Numero righe:", len(df))
    print("Lunghezza media recensione (caratteri):", round(df["text"].str.len().mean(), 2))
    print("Lunghezza media recensione (parole):", round(df["text"].str.split().str.len().mean(), 2))

In [ ]:
train_df = load_fasttext_dataframe(
    TRAIN_PATH,
    sample_size=TRAIN_SAMPLE_SIZE,
    seed=RANDOM_STATE,
)

test_df = load_fasttext_dataframe(
    TEST_PATH,
    sample_size=TEST_SAMPLE_SIZE,
    seed=RANDOM_STATE + 1,
)

display(train_df.head())
summarize_dataset(train_df, "Train sample")
summarize_dataset(test_df, "Test sample")

### Filtro lingua opzionale

Se vuoi allineare il progetto a un caso d'uso **strettamente italiano**, puoi attivare il filtro lingua.

Limiti pratici:

- richiede `langdetect`
- su `100k` recensioni puo' essere lento
- dato che il dataset Kaggle e' prevalentemente inglese, il numero di recensioni italiane potrebbe essere ridotto

In [ ]:
def filter_by_language(df, language="it", text_col="text", min_chars=20):
    if not LANGDETECT_AVAILABLE:
        raise ImportError("Installa langdetect con: python -m pip install langdetect")

    detected_languages = []
    for text in df[text_col].fillna(""):
        text = str(text)
        if len(text) < min_chars:
            detected_languages.append("unknown")
            continue

        try:
            detected_languages.append(detect(text))
        except LangDetectException:
            detected_languages.append("unknown")

    filtered_df = df.copy()
    filtered_df["detected_language"] = detected_languages
    filtered_df = filtered_df[filtered_df["detected_language"] == language].reset_index(drop=True)
    return filtered_df

if ENABLE_LANGUAGE_FILTER:
    train_df = filter_by_language(train_df, language=TARGET_LANGUAGE)
    test_df = filter_by_language(test_df, language=TARGET_LANGUAGE)
    print("Filtro lingua attivato.")
    summarize_dataset(train_df, "Train sample filtrato")
    summarize_dataset(test_df, "Test sample filtrato")
else:
    print("Filtro lingua disabilitato. Si procede con il campione cosi' come caricato.")

## 2. Text preprocessing

Il preprocessing richiesto comprende:

- conversione in minuscolo
- rimozione di URL
- rimozione di punteggiatura e numeri
- rimozione di stopword italiane
- stemming opzionale

Per mantenere la pipeline pulita e compatibile con `GridSearchCV`, implementiamo un trasformatore custom `TextCleaner`.

In [ ]:
ITALIAN_STOPWORDS = frozenset(
    {
        "a", "ad", "al", "allo", "ai", "agli", "all", "agl", "alla", "alle",
        "anche", "ancora", "avere", "aveva", "avevano", "ben", "buono", "che",
        "chi", "cinque", "ci", "coi", "col", "come", "con", "contro", "cui",
        "da", "dagli", "dai", "dal", "dall", "dalla", "dalle", "davanti", "de",
        "degli", "dei", "del", "dell", "della", "delle", "dentro", "di", "dopo",
        "due", "e", "ecco", "ed", "era", "erano", "essere", "fa", "faccio",
        "fare", "fatto", "fino", "fra", "gli", "ha", "hai", "hanno", "ho", "i",
        "il", "in", "indietro", "invece", "io", "la", "le", "lei", "li", "lo",
        "loro", "lui", "ma", "me", "meglio", "mi", "mia", "mie", "miei", "mila",
        "mio", "molta", "molti", "molto", "nei", "nella", "nelle", "nello",
        "negli", "nei", "no", "noi", "non", "nostra", "nostre", "nostri",
        "nostro", "nove", "nuovi", "nuovo", "o", "oltre", "ora", "otto", "peggio",
        "pero", "persino", "piu", "poco", "prima", "puo", "pure", "quale",
        "quanta", "quante", "quanti", "quanto", "quattro", "quella", "quelle",
        "quelli", "quello", "questa", "queste", "questi", "questo", "qui",
        "quindi", "sarai", "sara", "saranno", "se", "sei", "sembrava", "sempre",
        "senza", "sette", "si", "sia", "siamo", "siete", "solo", "sono", "sopra",
        "soprattutto", "sotto", "sta", "stai", "stanno", "sto", "su", "sua",
        "subito", "sue", "sugli", "sui", "sul", "sulla", "sulle", "sullo", "suo",
        "suoi", "tanto", "te", "tempo", "tra", "tre", "triplo", "tu", "tua",
        "tue", "tuo", "tuoi", "tutti", "tutto", "un", "una", "uno", "va", "vai",
        "voi", "volte", "vostra", "vostre", "vostri", "vostro",
    }
)

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
NON_LETTER_PATTERN = re.compile(r"[^a-zà-ÿ\s]")
MULTISPACE_PATTERN = re.compile(r"\s+")

def normalize_text(text):
    if not isinstance(text, str):
        if pd.isna(text):
            text = ""
        else:
            text = str(text)

    text = text.lower()
    text = URL_PATTERN.sub(" ", text)
    text = text.replace("\n", " ").replace("\r", " ")
    text = NON_LETTER_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

class TextCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, stopwords=ITALIAN_STOPWORDS, use_stemming=False, min_token_length=2):
        self.stopwords = stopwords
        self.use_stemming = use_stemming
        self.min_token_length = min_token_length

    def fit(self, X, y=None):
        self._stemmer = None
        if self.use_stemming:
            if not STEMMER_AVAILABLE:
                raise ImportError("Per usare lo stemming installa nltk: python -m pip install nltk")
            self._stemmer = SnowballStemmer("italian")
        return self

    def transform(self, X):
        cleaned_texts = []
        series = pd.Series(X).fillna("")

        for text in series:
            normalized = normalize_text(text)
            tokens = [
                token
                for token in normalized.split()
                if len(token) >= self.min_token_length and token not in self.stopwords
            ]

            if self._stemmer is not None:
                tokens = [self._stemmer.stem(token) for token in tokens]

            cleaned_texts.append(" ".join(tokens))

        return cleaned_texts

preview_cleaner = TextCleaner(use_stemming=False)
preview_df = train_df.loc[:4, ["sentiment", "text"]].copy()
preview_df["text_clean"] = preview_cleaner.fit_transform(preview_df["text"])
display(preview_df)

In [ ]:
eda_cleaner = TextCleaner(use_stemming=False)
eda_df = train_df.copy()
eda_df["text_clean"] = eda_cleaner.fit_transform(eda_df["text"])
eda_df["review_len_chars"] = eda_df["text"].str.len()
eda_df["review_len_words"] = eda_df["text"].str.split().str.len()

display(eda_df[["sentiment", "text", "text_clean", "review_len_chars", "review_len_words"]].head())

## 3. Exploratory Data Analysis (EDA)

Analizziamo:

- distribuzione delle classi
- distribuzione della lunghezza delle recensioni
- word cloud per classe
- parole piu' frequenti nei testi positivi e negativi

In [ ]:
class_distribution = (
    eda_df["sentiment"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)
class_distribution["share_pct"] = (100 * class_distribution["count"] / len(eda_df)).round(2)
display(class_distribution)

plt.figure(figsize=(8, 5))
sns.barplot(data=class_distribution, x="sentiment", y="count", palette="Set2")
plt.title("Distribuzione delle classi nel train sample")
plt.xlabel("Sentiment")
plt.ylabel("Numero recensioni")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(
    data=eda_df,
    x="review_len_chars",
    hue="sentiment",
    bins=60,
    kde=True,
    ax=axes[0],
    element="step",
)
axes[0].set_title("Distribuzione lunghezza recensioni (caratteri)")
axes[0].set_xlabel("Numero caratteri")

sns.histplot(
    data=eda_df,
    x="review_len_words",
    hue="sentiment",
    bins=60,
    kde=True,
    ax=axes[1],
    element="step",
)
axes[1].set_title("Distribuzione lunghezza recensioni (parole)")
axes[1].set_xlabel("Numero parole")

plt.tight_layout()
plt.show()

In [ ]:
def plot_wordcloud(texts, title, colormap):
    if not WORDCLOUD_AVAILABLE:
        raise ImportError("Installa wordcloud con: python -m pip install wordcloud")

    joined_text = " ".join(texts)
    if not joined_text.strip():
        print(f"Nessun testo disponibile per: {title}")
        return

    cloud = WordCloud(
        width=1200,
        height=600,
        background_color="white",
        colormap=colormap,
    ).generate(joined_text)

    plt.figure(figsize=(12, 5))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(title)
    plt.show()

plot_wordcloud(
    eda_df.loc[eda_df["target"] == 1, "text_clean"],
    title="Word Cloud - recensioni positive",
    colormap="Greens",
)

plot_wordcloud(
    eda_df.loc[eda_df["target"] == 0, "text_clean"],
    title="Word Cloud - recensioni negative",
    colormap="Reds",
)

In [ ]:
def get_top_words(df, target_value, text_col="text_clean", top_n=20):
    tokens = " ".join(df.loc[df["target"] == target_value, text_col]).split()
    counts = Counter(tokens).most_common(top_n)
    return pd.DataFrame(counts, columns=["word", "frequency"])

top_positive_words = get_top_words(eda_df, target_value=1, top_n=20)
top_negative_words = get_top_words(eda_df, target_value=0, top_n=20)

display(top_positive_words)
display(top_negative_words)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sns.barplot(
    data=top_positive_words.sort_values("frequency"),
    x="frequency",
    y="word",
    color="forestgreen",
    ax=axes[0],
)
axes[0].set_title("Parole piu' frequenti - positivo")

sns.barplot(
    data=top_negative_words.sort_values("frequency"),
    x="frequency",
    y="word",
    color="firebrick",
    ax=axes[1],
)
axes[1].set_title("Parole piu' frequenti - negativo")

plt.tight_layout()
plt.show()

## 4. Vectorization & Modeling

Costruiamo una pipeline completa:

`testo raw -> TextCleaner -> TfidfVectorizer -> classificatore`

La pipeline evita leakage:

- il `train_test_split` viene fatto **prima** del fitting
- `TF-IDF` viene fit-tato solo sul training set
- lo stesso preprocessing viene riutilizzato in validazione, test e deployment

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    train_df["text"],
    train_df["target"],
    test_size=VALID_SIZE,
    stratify=train_df["target"],
    random_state=RANDOM_STATE,
)

X_test = test_df["text"].copy()
y_test = test_df["target"].copy()

print("Dimensione training set:", X_train.shape[0])
print("Dimensione validation set:", X_valid.shape[0])
print("Dimensione test set:", X_test.shape[0])
print("Positive rate train:", round(y_train.mean(), 4))
print("Positive rate validation:", round(y_valid.mean(), 4))
print("Positive rate test:", round(y_test.mean(), 4))

In [ ]:
def build_sentiment_pipeline(classifier, use_stemming=False):
    return Pipeline(
        steps=[
            ("cleaner", TextCleaner(use_stemming=use_stemming)),
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=False,
                    max_features=40_000,
                    ngram_range=(1, 2),
                    min_df=3,
                    max_df=0.95,
                    sublinear_tf=True,
                    dtype=np.float32,
                ),
            ),
            ("classifier", classifier),
        ]
    )

def get_score_values(model, X):
    if hasattr(model, "predict_proba"):
        return np.asarray(model.predict_proba(X))[:, 1]

    if hasattr(model, "decision_function"):
        return np.asarray(model.decision_function(X)).ravel()

    raise AttributeError("Il modello non espone predict_proba o decision_function.")

def score_to_probability(score_values):
    score_values = np.asarray(score_values, dtype=float).ravel()
    if np.min(score_values) < 0 or np.max(score_values) > 1:
        clipped = np.clip(score_values, -50, 50)
        return 1.0 / (1.0 + np.exp(-clipped))
    return score_values

def evaluate_model(model, X, y, split_name):
    y_pred = model.predict(X)
    y_score = get_score_values(model, X)

    metrics = {
        "split": split_name,
        "accuracy": accuracy_score(y, y_pred),
        "f1": f1_score(y, y_pred),
        "roc_auc": roc_auc_score(y, y_score),
    }
    return metrics, y_pred, y_score

In [ ]:
baseline_models = {
    "LogisticRegression": LogisticRegression(max_iter=2000, solver="liblinear"),
    "MultinomialNB": MultinomialNB(alpha=1.0),
    "LinearSVC": LinearSVC(C=1.0),
    "SGDClassifier": SGDClassifier(
        loss="log_loss",
        alpha=1e-4,
        penalty="l2",
        max_iter=2000,
        random_state=RANDOM_STATE,
    ),
}

baseline_results = []
baseline_pipelines = {}

for model_name, estimator in baseline_models.items():
    pipeline = build_sentiment_pipeline(estimator, use_stemming=False)
    pipeline.fit(X_train, y_train)
    metrics, _, _ = evaluate_model(pipeline, X_valid, y_valid, split_name="validation")
    metrics["model"] = model_name
    baseline_results.append(metrics)
    baseline_pipelines[model_name] = pipeline

baseline_results_df = (
    pd.DataFrame(baseline_results)
    .sort_values(["f1", "roc_auc", "accuracy"], ascending=False)
    .reset_index(drop=True)
)

display(
    baseline_results_df.style.format(
        {
            "accuracy": "{:.4f}",
            "f1": "{:.4f}",
            "roc_auc": "{:.4f}",
        }
    )
)

### Tuning con GridSearchCV

Ora ottimizziamo simultaneamente:

- iperparametri di `TF-IDF`
- scelta del classificatore
- iperparametri del classificatore

Nota pratica:

- questa cella puo' richiedere tempo su `100k` righe
- se vuoi ridurre il runtime, restringi il `param_grid`

In [ ]:
STEMMING_OPTIONS = [False]

grid_pipeline = Pipeline(
    steps=[
        ("cleaner", TextCleaner()),
        ("tfidf", TfidfVectorizer(lowercase=False, dtype=np.float32)),
        ("classifier", LogisticRegression(max_iter=2000, solver="liblinear")),
    ]
)

param_grid = [
    {
        "cleaner__use_stemming": STEMMING_OPTIONS,
        "tfidf__max_features": [30_000, 60_000],
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "tfidf__min_df": [2, 5],
        "tfidf__max_df": [0.90, 0.98],
        "tfidf__sublinear_tf": [True],
        "classifier": [LogisticRegression(max_iter=2000, solver="liblinear")],
        "classifier__C": [0.5, 1.0, 2.0],
    },
    {
        "cleaner__use_stemming": STEMMING_OPTIONS,
        "tfidf__max_features": [30_000, 60_000],
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "tfidf__min_df": [2, 5],
        "tfidf__max_df": [0.90, 0.98],
        "tfidf__sublinear_tf": [True],
        "classifier": [MultinomialNB()],
        "classifier__alpha": [0.3, 0.7, 1.0],
    },
    {
        "cleaner__use_stemming": STEMMING_OPTIONS,
        "tfidf__max_features": [30_000, 60_000],
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "tfidf__min_df": [2, 5],
        "tfidf__max_df": [0.90, 0.98],
        "tfidf__sublinear_tf": [True],
        "classifier": [LinearSVC()],
        "classifier__C": [0.5, 1.0, 2.0],
    },
    {
        "cleaner__use_stemming": STEMMING_OPTIONS,
        "tfidf__max_features": [30_000, 60_000],
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "tfidf__min_df": [2, 5],
        "tfidf__max_df": [0.90, 0.98],
        "tfidf__sublinear_tf": [True],
        "classifier": [
            SGDClassifier(
                loss="log_loss",
                max_iter=2000,
                random_state=RANDOM_STATE,
            )
        ],
        "classifier__alpha": [1e-5, 1e-4, 1e-3],
        "classifier__penalty": ["l2", "elasticnet"],
    },
]

scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=grid_pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit="f1",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=2,
    return_train_score=False,
)

grid_search.fit(X_train, y_train)

print("Best CV F1-score:", round(grid_search.best_score_, 4))
print("Best params:", grid_search.best_params_)

In [ ]:
cv_results_df = pd.DataFrame(grid_search.cv_results_).copy()
cv_results_df["model_family"] = cv_results_df["param_classifier"].astype(str)

cols_to_display = [
    "rank_test_f1",
    "model_family",
    "mean_test_accuracy",
    "mean_test_f1",
    "mean_test_roc_auc",
    "param_tfidf__max_features",
    "param_tfidf__ngram_range",
    "param_tfidf__min_df",
    "param_tfidf__max_df",
    "param_classifier__C",
    "param_classifier__alpha",
    "param_classifier__penalty",
]

display(
    cv_results_df[cols_to_display]
    .sort_values("rank_test_f1")
    .head(10)
    .style.format(
        {
            "mean_test_accuracy": "{:.4f}",
            "mean_test_f1": "{:.4f}",
            "mean_test_roc_auc": "{:.4f}",
        }
    )
)

## 5. Evaluation & Error Analysis

Per il miglior modello:

- valutiamo performance su validation e test set
- generiamo confusion matrix e curva ROC
- estraiamo le parole piu' influenti
- analizziamo gli errori piu' ambigui

In [ ]:
best_pipeline = grid_search.best_estimator_

validation_metrics, y_valid_pred, y_valid_score = evaluate_model(
    best_pipeline,
    X_valid,
    y_valid,
    split_name="validation",
)

final_pipeline = clone(best_pipeline)
final_pipeline.fit(train_df["text"], train_df["target"])

test_metrics, y_test_pred, y_test_score = evaluate_model(
    final_pipeline,
    X_test,
    y_test,
    split_name="official_test_sample",
)

metrics_summary_df = pd.DataFrame([validation_metrics, test_metrics])
display(
    metrics_summary_df.style.format(
        {
            "accuracy": "{:.4f}",
            "f1": "{:.4f}",
            "roc_auc": "{:.4f}",
        }
    )
)

print("Classification report - validation")
print(classification_report(y_valid, y_valid_pred, target_names=["negativo", "positivo"]))

print("Classification report - official test sample")
print(classification_report(y_test, y_test_pred, target_names=["negativo", "positivo"]))

In [ ]:
goal_check_df = pd.DataFrame(
    [
        {
            "metric": "accuracy",
            "target": TARGET_METRICS["accuracy"],
            "validation_value": validation_metrics["accuracy"],
            "test_value": test_metrics["accuracy"],
        },
        {
            "metric": "f1",
            "target": TARGET_METRICS["f1"],
            "validation_value": validation_metrics["f1"],
            "test_value": test_metrics["f1"],
        },
        {
            "metric": "roc_auc",
            "target": TARGET_METRICS["roc_auc"],
            "validation_value": validation_metrics["roc_auc"],
            "test_value": test_metrics["roc_auc"],
        },
    ]
)
goal_check_df["validation_target_met"] = goal_check_df["validation_value"] >= goal_check_df["target"]
goal_check_df["test_target_met"] = goal_check_df["test_value"] >= goal_check_df["target"]
display(
    goal_check_df.style.format(
        {
            "target": "{:.2f}",
            "validation_value": "{:.4f}",
            "test_value": "{:.4f}",
        }
    )
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    display_labels=["negativo", "positivo"],
    cmap="Blues",
    colorbar=False,
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix - official test sample")

RocCurveDisplay.from_predictions(y_test, y_test_score, ax=axes[1])
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[1].set_title("ROC Curve - official test sample")

plt.tight_layout()
plt.show()

In [ ]:
def extract_feature_importance(pipeline, top_n=20):
    vectorizer = pipeline.named_steps["tfidf"]
    classifier = pipeline.named_steps["classifier"]
    feature_names = np.asarray(vectorizer.get_feature_names_out())

    if hasattr(classifier, "coef_"):
        weights = classifier.coef_[0]
    elif hasattr(classifier, "feature_log_prob_"):
        weights = classifier.feature_log_prob_[1] - classifier.feature_log_prob_[0]
    else:
        raise ValueError("Il classificatore selezionato non espone coefficienti interpretabili.")

    importance_df = (
        pd.DataFrame({"term": feature_names, "weight": weights})
        .sort_values("weight", ascending=False)
        .reset_index(drop=True)
    )

    top_positive = importance_df.head(top_n).copy()
    top_negative = importance_df.tail(top_n).sort_values("weight").copy()
    return top_positive, top_negative, importance_df

top_positive_terms, top_negative_terms, importance_df = extract_feature_importance(
    final_pipeline,
    top_n=20,
)

display(top_positive_terms)
display(top_negative_terms)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sns.barplot(
    data=top_positive_terms.sort_values("weight"),
    x="weight",
    y="term",
    color="seagreen",
    ax=axes[0],
)
axes[0].set_title("Termini piu' associati al sentiment positivo")

sns.barplot(
    data=top_negative_terms.sort_values("weight", ascending=False),
    x="weight",
    y="term",
    color="indianred",
    ax=axes[1],
)
axes[1].set_title("Termini piu' associati al sentiment negativo")

plt.tight_layout()
plt.show()

In [ ]:
def build_error_analysis_frame(model, X, y_true, raw_texts, top_n=15):
    y_true_series = pd.Series(y_true).reset_index(drop=True)
    raw_texts_series = pd.Series(raw_texts).reset_index(drop=True)
    y_pred = pd.Series(model.predict(X)).reset_index(drop=True)
    score_values = get_score_values(model, X)
    prob_positive = score_to_probability(score_values)

    error_df = pd.DataFrame(
        {
            "text": raw_texts_series,
            "y_true": y_true_series,
            "y_pred": y_pred,
            "prob_positive": prob_positive,
        }
    )
    error_df["true_label"] = error_df["y_true"].map({0: "negativo", 1: "positivo"})
    error_df["pred_label"] = error_df["y_pred"].map({0: "negativo", 1: "positivo"})
    error_df["ambiguity"] = (error_df["prob_positive"] - 0.5).abs()
    error_df["text_length_words"] = error_df["text"].str.split().str.len()

    error_df = error_df[error_df["y_true"] != error_df["y_pred"]].copy()
    error_df = error_df.sort_values(["ambiguity", "text_length_words"], ascending=[True, True])
    return error_df.head(top_n).reset_index(drop=True)

ambiguous_errors_df = build_error_analysis_frame(
    final_pipeline,
    X_test,
    y_test,
    raw_texts=test_df["text"],
    top_n=15,
)

display(
    ambiguous_errors_df[
        ["true_label", "pred_label", "prob_positive", "ambiguity", "text_length_words", "text"]
    ]
)

## 6. Deployment

Salviamo la pipeline finale con `joblib` e definiamo una funzione di inferenza semplice:

- input: testo raw
- output: sentiment predetto + probabilita'

In [ ]:
FINAL_MODEL_PATH = ARTIFACTS_DIR / "amazon_sentiment_tfidf_pipeline.joblib"

joblib.dump(final_pipeline, FINAL_MODEL_PATH)
print("Pipeline salvata in:", FINAL_MODEL_PATH.resolve())

def analizza_sentiment(testo):
    if not isinstance(testo, str) or not testo.strip():
        raise ValueError("Il testo da analizzare non puo' essere vuoto.")

    pred = int(final_pipeline.predict([testo])[0])
    score = float(get_score_values(final_pipeline, [testo])[0])
    prob_positive = float(score_to_probability([score])[0])

    return {
        "testo": testo,
        "classe_predetta": pred,
        "sentiment_predetto": "positivo" if pred == 1 else "negativo",
        "probabilita_positiva": round(prob_positive, 4),
        "probabilita_negativa": round(1.0 - prob_positive, 4),
    }

analizza_sentiment("Consegna velocissima, prodotto eccellente e ben confezionato.")